In [ ]:
import joblib
import pandas as pd
from xgboost import XGBClassifier, XGBRegressor
from lightgbm import LGBMClassifier, LGBMRegressor
from catboost import CatBoostClassifier, CatBoostRegressor
from sklearn.linear_model import SGDClassifier, SGDRegressor, LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.ensemble import AdaBoostClassifier, AdaBoostRegressor 
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.ensemble import ExtraTreesClassifier, ExtraTreesRegressor
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import roc_auc_score
from pathlib import Path
import datetime as dt
import time
import pytz
import json
import os

In [ ]:
pst = pytz.timezone('America/Los_Angeles')
dt_str = dt.datetime.now(pst).strftime("%Y-%m-%d-%I_%M_%S_%p")
dt_str

In [ ]:
experiment_config = {
    "experiment": {
        "id": f"{dt_str}_adaboost",
        "model": "adaboost",
        "type": "baseline",
        "dataset_type": "features_ohe_imputed",
        "description": "basic adaboost baseline",
        "target": "classification",
    },

    "cv": {
        "strategy": "StratifiedKFold",
        "n_splits": 5,
        "shuffle": True,
        "random_state": 0
    },

    "params": {
        "n_estimators": 1000,
        "learning_rate": 0.1
    }
}

In [ ]:
if os.environ.get('KAGGLE_KERNEL_RUN_TYPE'):
    data_path = "/kaggle/input/datasets/abhinavneelam/smartphone-addiction/data"
    output_path = "/kaggle/working/"
else:
    data_path = "../../data"
    output_path = "../../"

experiment_path = Path(output_path) / "experiments" / f"{dt_str}_{experiment_config["experiment"]["model"]}"
experiment_path.mkdir(parents=True, exist_ok=True)
experiment_path

In [ ]:
with open(experiment_path / "config.json", "w") as f:
    json.dump(experiment_config, f, indent=4)

In [ ]:
ss = pd.read_csv(f"{data_path}/raw/sample_submission.csv")
target_column = ss.columns[-1]
target_column

In [ ]:
raw_train_id = pd.read_csv(f"{data_path}/raw/train.csv")["id"]
X = pd.read_csv(f"{data_path}/processed/train_{experiment_config["experiment"]["dataset_type"]}.csv")
X_test = pd.read_csv(f"{data_path}/processed/test_{experiment_config["experiment"]["dataset_type"]}.csv")
y = pd.read_csv(f"{data_path}/processed/train_labels.csv")

X.info()

In [ ]:
cat_cols = X.select_dtypes(include=["object", "string"]).columns

for frame in [X, X_test]:
    for col in cat_cols:
        frame[col] = frame[col].astype('category')

In [ ]:
X.info()

In [ ]:
def make_model(config):
    name = config["experiment"]["model"]
    target = config["experiment"]["target"]
    params = config["params"]

    model_dict = {
        "adaboost": (AdaBoostClassifier, AdaBoostRegressor),
        "gradientboost": (GradientBoostingClassifier, GradientBoostingRegressor),
        "catboost": (CatBoostClassifier, CatBoostRegressor),
        "xgboost": (XGBClassifier, XGBRegressor),
        "lightgbm": (LGBMClassifier, LGBMRegressor),
        "randomforest": (RandomForestClassifier, RandomForestRegressor),
        "extratrees": (ExtraTreesClassifier, ExtraTreesRegressor),
        "hgbc": (HistGradientBoostingClassifier, HistGradientBoostingRegressor),
        "knn": (KNeighborsClassifier, KNeighborsRegressor),
        "sgd": (SGDClassifier, SGDRegressor),
        "linear": (LogisticRegression, LinearRegression),
        "decisiontree": (DecisionTreeClassifier, DecisionTreeRegressor),
        "mlp": (MLPClassifier, MLPRegressor),
    }

    target_index = target == "regression"
    model_class = model_dict[name][target_index]

    return model_class(**params)

In [ ]:
def oof_fit(config, model, X_train, y_train, X_valid, y_valid):
    name = config["experiment"]["model"]
    no_eval_models = ["adaboost", "gradientboost", "randomforest", "extratrees", "knn", "sgd", "linear", "decisiontree", "mlp"]
    fit_params = config["fit_params"]

    if name in no_eval_models:
        model.fit(X_train, y_train)
    elif name == "hgbc":
        model.fit(X_train, y_train, X_val=X_valid, y_val=y_valid)
    elif name == "lightgbm":
        model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], **fit_params)
    else:
        model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)])

In [ ]:
kf = StratifiedKFold(n_splits=5, random_state=0, shuffle=True)

y_cv = pd.Series(index=y.index, dtype=float, name=target_column)
fold_scores = []

start_time = time.time()

for train_index, valid_index in kf.split(X, y):
    X_train, X_valid = X.iloc[train_index], X.iloc[valid_index]
    y_train, y_valid = y.iloc[train_index], y.iloc[valid_index]

    model = make_model(experiment_config)
    oof_fit(experiment_config, model, X_train, y_train, X_valid, y_valid)

    y_pred = model.predict_proba(X_valid)[:, 1]

    y_cv.iloc[valid_index] = y_pred

    fold_auc_score = roc_auc_score(y_valid, y_pred)
    fold_scores.append(round(fold_auc_score, 5))

elapsed = time.time() - start_time

y_pred_df = pd.concat([raw_train_id, y_cv], axis=1)
y_pred_df.to_csv(experiment_path / "oof.csv", index=False)

In [ ]:
valid_auc_score = roc_auc_score(y, y_cv)

print("Validation AUC:", valid_auc_score)

In [ ]:
metrics = {
    "experiment": dt_str + f"_{experiment_config["experiment"]["model"]}",
    "model": f"{experiment_config["experiment"]["model"]}",
    "dataset_type": f"{experiment_config["experiment"]["dataset_type"]}",
    "cv": {
        "strategy": "StratifiedKFold",
        "n_splits": 5,
        "random_state": 0,
        "fold_scores": fold_scores,
        "mean": round(sum(fold_scores) / len(fold_scores), 5),
        "std": round(float(pd.Series(fold_scores).std(ddof=1)), 5)
    },
    "primary_metric": {
        "name": "auc",
        "value": round(valid_auc_score, 5)
    },
    "training": {
        "duration_seconds": round(elapsed, 2)
    }
}

with open(experiment_path / "metrics.json", "w") as f:
    json.dump(metrics, f, indent=4)

In [ ]:
best_iteration = model.best_iteration if hasattr(model, "best_iteration") else experiment_config["params"]["n_estimators"]

experiment_config["params"]["early_stopping_rounds"] = None
experiment_config["params"]["n_estimators"] = best_iteration

model = make_model(experiment_config)
model.fit(X, y)

joblib.dump(model, experiment_path / f"{experiment_config["experiment"]["model"]}.pkl")

In [ ]:
y_pred = model.predict_proba(X_test)[:, 1]
ss[target_column] = y_pred

ss.to_csv(experiment_path / f"{experiment_config["experiment"]["model"]}_submission.csv", index=False)
ss